# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


# Bandwidth Feature-Augmentation Experiment

This notebook evaluates whether adding `bandwidth` to the six-feature timestamp-free representation changes predictive performance. The experiment uses XGBoost and MLP under the frozen grouped-random and chronological partitions, with seeds 42, 43, and 44.

No absolute timestamp variables are included. Hyperparameters are inherited from the corresponding timestamp-free six-feature configurations; no additional hyperparameter search is performed.

In [ ]:
from pathlib import Path
import platform
import sys
import json
import hashlib

import numpy as np
import pandas as pd
import sklearn
import xgboost
import torch

REVISION_ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

DATASET_DIR = (
    REVISION_ROOT
    / "jpco_o3_splits_v1"
)

OUTPUT_DIR = (
    REVISION_ROOT
    / "bandwidth_augmentation_v1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    DATASET_DIR
    / "manifest.json"
)

PRIMARY_DATA_PATH = (
    DATASET_DIR
    / "o3_primary.csv.gz"
)

SPLIT_ASSIGNMENTS_PATH = (
    DATASET_DIR
    / "split_assignments.csv.gz"
)

SPLIT_SUMMARY_PATH = (
    DATASET_DIR
    / "split_summary.csv"
)

XGB_STAGE1_DIR = (
    REVISION_ROOT
    / "xgboost_stage1_results"
)

MLP_STAGE1_DIR = (
    REVISION_ROOT
    / "mlp_stage1_results"
)

XGB_AUTHORITATIVE_SCRIPT = (
    REVISION_ROOT
    / "xgboost_revision_stage1.py"
)

MLP_AUTHORITATIVE_SCRIPT = (
    REVISION_ROOT
    / "mlp_revision_stage1.py"
)

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Working directory:", Path.cwd())
print("Revision root:", REVISION_ROOT)
print("Dataset directory:", DATASET_DIR)
print("Output directory:", OUTPUT_DIR)

print()
print("--- Package versions ---")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("torch:", torch.__version__)

required_paths = [
    MANIFEST_PATH,
    PRIMARY_DATA_PATH,
    SPLIT_ASSIGNMENTS_PATH,
    SPLIT_SUMMARY_PATH,
    XGB_STAGE1_DIR,
    MLP_STAGE1_DIR,
    XGB_AUTHORITATIVE_SCRIPT,
    MLP_AUTHORITATIVE_SCRIPT,
]

print()
print("--- Required paths ---")

for path in required_paths:
    status = (
        "FOUND"
        if path.exists()
        else "MISSING"
    )

    print(f"{status}: {path}")

    assert path.exists(), (
        f"Required path is missing: {path}"
    )

print()
print("ENVIRONMENT AND PATH PREFLIGHT PASSED")

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


manifest = json.loads(
    MANIFEST_PATH.read_text()
)

print("--- Manifest ---")
print("Version:", manifest["version"])
print("Primary rows:", manifest["primary_rows"])
print("Target:", manifest["target"])

print()
print("--- Feature sets ---")

for name, features in (
    manifest["feature_sets"].items()
):
    print(f"{name}: {features}")

TIMESTAMP_FREE_SIX = [
    "duration",
    "peak_frequency",
    "central_freq",
    "amplitude",
    "snr",
    "q_value",
]

TIMESTAMP_FREE_PLUS_BANDWIDTH = [
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

TARGET_COLUMN = "ml_label"

assert (
    manifest["feature_sets"][
        "timestamp_free_six"
    ]
    == TIMESTAMP_FREE_SIX
)

assert (
    "bandwidth"
    in manifest["feature_sets"][
        "enriched_eleven"
    ]
)

required_primary_columns = (
    TIMESTAMP_FREE_PLUS_BANDWIDTH
    + [TARGET_COLUMN]
)

primary_header = pd.read_csv(
    PRIMARY_DATA_PATH,
    nrows=0,
).columns.tolist()

print()
print("Primary-data columns:")
print(primary_header)

missing_columns = sorted(
    set(required_primary_columns)
    - set(primary_header)
)

print()
print(
    "Missing required columns:",
    missing_columns,
)

assert not missing_columns

print()
print(
    "BANDWIDTH FEATURE AVAILABILITY VERIFIED"
)

In [ ]:
EXPECTED_INPUT_HASHES = {
    "manifest.json": (
        "f1eedfaa282d6ae90e1567d271f9baff"
        "06c62ba233aff5890ae5dad85cfe9605"
    ),
    "o3_primary.csv.gz": (
        "9d7d7d217563ccb121eba466fc442d83"
        "d7f25495e6e5ba40ea9a0962114829c6"
    ),
    "split_summary.csv": (
        "61de488a44179ca86205ffcb45e98baa"
        "9eb2d9a7b52a1ab13cf2845368f88d01"
    ),
}

INPUT_PATHS = {
    "manifest.json": MANIFEST_PATH,
    "o3_primary.csv.gz": (
        PRIMARY_DATA_PATH
    ),
    "split_summary.csv": (
        SPLIT_SUMMARY_PATH
    ),
}

observed_hashes = {}

print("--- Frozen input hashes ---")

for filename, path in INPUT_PATHS.items():
    observed = sha256_file(path)
    expected = EXPECTED_INPUT_HASHES[
        filename
    ]

    observed_hashes[
        filename
    ] = observed

    print(filename)
    print("  observed:", observed)
    print("  expected:", expected)

    assert observed == expected, (
        f"Frozen input changed: {filename}"
    )

print()
print("FROZEN INPUT HASHES VERIFIED")

In [ ]:
split_summary = pd.read_csv(
    SPLIT_SUMMARY_PATH
)

display(split_summary)

expected_configurations = {
    "random_split",
    "chronological_split",
}

expected_partitions = {
    "train",
    "validation",
    "test",
}

observed_configurations = set(
    split_summary["configuration"]
)

assert (
    observed_configurations
    == expected_configurations
)

for configuration in (
    expected_configurations
):
    configuration_partitions = set(
        split_summary.loc[
            split_summary[
                "configuration"
            ]
            == configuration,
            "partition",
        ]
    )

    assert (
        configuration_partitions
        == expected_partitions
    )

print(
    "Random and chronological split "
    "definitions verified."
)

print()
print("SPLIT PREFLIGHT PASSED")

In [ ]:
CONDITIONS = [
    "random_split",
    "chronological_split",
]

selected_configurations = {}

for model, stage1_directory in {
    "XGBoost": XGB_STAGE1_DIR,
    "MLP": MLP_STAGE1_DIR,
}.items():

    selected_configurations[
        model
    ] = {}

    for split_name in CONDITIONS:
        selection_path = (
            stage1_directory
            / (
                f"{split_name}"
                "__timestamp_free_six"
            )
            / "selected_configuration.json"
        )

        assert selection_path.exists(), (
            "Missing selected configuration: "
            f"{selection_path}"
        )

        selection = json.loads(
            selection_path.read_text()
        )

        selected_configurations[
            model
        ][split_name] = {
            "path": str(selection_path),
            "sha256": sha256_file(
                selection_path
            ),
            "selection": selection,
        }

        print()
        print("MODEL:", model)
        print("SPLIT:", split_name)
        print("SOURCE FEATURE SET: timestamp_free_six")
        print("PATH:", selection_path)
        print(
            "SHA-256:",
            sha256_file(
                selection_path
            ),
        )

        if "parameters" in selection:
            print(
                "PARAMETERS:",
                selection["parameters"],
            )
        else:
            print(
                json.dumps(
                    selection,
                    indent=2,
                )
            )

print()
print(
    "AUTHORITATIVE TIMESTAMP-FREE "
    "CONFIGURATIONS LOCATED"
)

In [ ]:
PREFLIGHT_REPORT_PATH = (
    OUTPUT_DIR
    / "preflight_report.json"
)

preflight_report = {
    "experiment": (
        "timestamp_free_plus_bandwidth"
    ),
    "training_performed": False,
    "models": [
        "XGBoost",
        "MLP",
    ],
    "splits": CONDITIONS,
    "seeds": [
        42,
        43,
        44,
    ],
    "fit_count_planned": 12,
    "baseline_features": (
        TIMESTAMP_FREE_SIX
    ),
    "augmented_features": (
        TIMESTAMP_FREE_PLUS_BANDWIDTH
    ),
    "added_feature": "bandwidth",
    "absolute_timestamps_included": False,
    "hyperparameter_policy": (
        "Reuse the configuration selected "
        "for the corresponding model, split, "
        "and timestamp-free six-feature "
        "condition. No new search."
    ),
    "input_sha256": observed_hashes,
    "authoritative_scripts": {
        "XGBoost": {
            "path": str(
                XGB_AUTHORITATIVE_SCRIPT
            ),
            "sha256": sha256_file(
                XGB_AUTHORITATIVE_SCRIPT
            ),
        },
        "MLP": {
            "path": str(
                MLP_AUTHORITATIVE_SCRIPT
            ),
            "sha256": sha256_file(
                MLP_AUTHORITATIVE_SCRIPT
            ),
        },
    },
    "selected_configurations": (
        selected_configurations
    ),
}

PREFLIGHT_REPORT_PATH.write_text(
    json.dumps(
        preflight_report,
        indent=2,
        allow_nan=False,
    )
    + "\n"
)

print("SAVED:")
print(PREFLIGHT_REPORT_PATH)
print(
    "SHA-256:",
    sha256_file(
        PREFLIGHT_REPORT_PATH
    ),
)

print()
print(
    "BANDWIDTH-AUGMENTATION "
    "PREFLIGHT COMPLETE"
)
print("NO TRAINING PERFORMED")

In [ ]:
%%writefile bandwidth_augmentation_xgb_pilot.py

from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    f1_score,
)


FEATURES = [
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

TARGET = "ml_label"
PARTITION_COLUMN = "random_split"

EXPECTED_INPUT_HASHES = {
    "manifest.json": (
        "f1eedfaa282d6ae90e1567d271f9baff"
        "06c62ba233aff5890ae5dad85cfe9605"
    ),
    "o3_primary.csv.gz": (
        "9d7d7d217563ccb121eba466fc442d83"
        "d7f25495e6e5ba40ea9a0962114829c6"
    ),
    "split_summary.csv": (
        "61de488a44179ca86205ffcb45e98baa"
        "9eb2d9a7b52a1ab13cf2845368f88d01"
    ),
    "selected_configuration.json": (
        "16018fc9e88becd36ac31eabbe913ab41"
        "c277445c0f319bd14d25bb6e5c1c60f"
    ),
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def save_json(path: Path, value: dict) -> None:
    path.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )


def main() -> None:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--root",
        required=True,
        type=Path,
    )

    args = parser.parse_args()
    root = args.root.expanduser().resolve()

    dataset_dir = root / "jpco_o3_splits_v1"

    manifest_path = (
        dataset_dir / "manifest.json"
    )
    primary_path = (
        dataset_dir / "o3_primary.csv.gz"
    )
    split_summary_path = (
        dataset_dir / "split_summary.csv"
    )

    selection_path = (
        root
        / "xgboost_stage1_results"
        / "random_split__timestamp_free_six"
        / "selected_configuration.json"
    )

    output_dir = (
        root
        / "bandwidth_augmentation_v1"
        / "pilot"
        / "random_split__timestamp_free_plus_bandwidth"
        / "xgboost_seed_42"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    input_paths = {
        "manifest.json": manifest_path,
        "o3_primary.csv.gz": primary_path,
        "split_summary.csv": split_summary_path,
        "selected_configuration.json": (
            selection_path
        ),
    }

    print("Verifying frozen inputs...")

    observed_hashes = {}

    for name, path in input_paths.items():
        if not path.is_file():
            raise FileNotFoundError(path)

        observed = sha256_file(path)
        expected = EXPECTED_INPUT_HASHES[name]

        print(name)
        print("  observed:", observed)
        print("  expected:", expected)

        if observed != expected:
            raise RuntimeError(
                f"Input hash mismatch: {name}"
            )

        observed_hashes[name] = observed

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    selected = json.loads(
        selection_path.read_text(
            encoding="utf-8"
        )
    )

    assert (
        selected["selection_used_outer_validation"]
        is False
    )

    assert selected["features"] == [
        "duration",
        "peak_frequency",
        "central_freq",
        "amplitude",
        "snr",
        "q_value",
    ]

    labels = selected["labels"]

    assert len(labels) == 24
    assert len(set(labels)) == 24

    label_to_index = {
        label: index
        for index, label in enumerate(labels)
    }

    required_columns = (
        FEATURES
        + [
            TARGET,
            PARTITION_COLUMN,
        ]
    )

    print()
    print(
        "Loading grouped-random training "
        "and validation partitions..."
    )

    frame = pd.read_csv(
        primary_path,
        usecols=required_columns,
        float_precision="round_trip",
    )

    assert not frame[
        FEATURES + [TARGET]
    ].isna().any().any()

    observed_partitions = set(
        frame[PARTITION_COLUMN].unique()
    )

    assert observed_partitions == {
        "train",
        "validation",
        "test",
    }

    train = frame.loc[
        frame[PARTITION_COLUMN] == "train"
    ].copy()

    validation = frame.loc[
        frame[PARTITION_COLUMN]
        == "validation"
    ].copy()

    # The test partition is deliberately
    # not materialized or evaluated.
    del frame

    assert len(train) == 280_163
    assert len(validation) == 70_040

    unknown_train_labels = (
        set(train[TARGET].unique())
        - set(labels)
    )
    unknown_validation_labels = (
        set(validation[TARGET].unique())
        - set(labels)
    )

    assert not unknown_train_labels
    assert not unknown_validation_labels

    X_train = train[FEATURES].to_numpy(
        dtype=np.float32,
        copy=True,
    )
    X_validation = validation[
        FEATURES
    ].to_numpy(
        dtype=np.float32,
        copy=True,
    )

    y_train = (
        train[TARGET]
        .map(label_to_index)
        .to_numpy(dtype=np.int32)
    )

    y_validation = (
        validation[TARGET]
        .map(label_to_index)
        .to_numpy(dtype=np.int32)
    )

    del train
    del validation

    parameters = dict(selected["params"])

    parameters.update(
        {
            "objective": "multi:softprob",
            "num_class": len(labels),
            "tree_method": "hist",
            "device": "cpu",
            "nthread": 2,
            "seed": 42,
            "eval_metric": "mlogloss",
            "verbosity": 1,
        }
    )

    rounds = int(selected["rounds"])

    protocol = {
        "experiment": (
            "timestamp_free_plus_bandwidth"
        ),
        "model": "XGBoost",
        "split": "random_split",
        "seed": 42,
        "training_performed": True,
        "test_evaluated": False,
        "source_configuration": str(
            selection_path
        ),
        "source_configuration_sha256": (
            observed_hashes[
                "selected_configuration.json"
            ]
        ),
        "source_feature_set": (
            "timestamp_free_six"
        ),
        "features": FEATURES,
        "feature_count": len(FEATURES),
        "target": TARGET,
        "partition_column": (
            PARTITION_COLUMN
        ),
        "train_rows": len(X_train),
        "validation_rows": len(
            X_validation
        ),
        "labels": labels,
        "parameters": parameters,
        "rounds": rounds,
        "input_sha256": observed_hashes,
        "manifest_version": manifest[
            "version"
        ],
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scikit_learn": (
                sklearn.__version__
            ),
            "xgboost": xgb.__version__,
        },
    }

    protocol_path = (
        output_dir / "protocol.json"
    )

    save_json(
        protocol_path,
        protocol,
    )

    protocol_sha256 = sha256_file(
        protocol_path
    )

    print()
    print(
        f"Train: {len(X_train):,}; "
        f"validation: "
        f"{len(X_validation):,}; "
        f"features: {len(FEATURES)}"
    )
    print(
        f"CPU threads: "
        f"{parameters['nthread']}; "
        f"rounds: {rounds}; "
        f"maximum depth: "
        f"{parameters['max_depth']}"
    )
    print(
        "Test partition will not be evaluated."
    )
    print(
        "Training starts; progress prints "
        "every 20 rounds."
    )

    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        feature_names=FEATURES,
    )

    dvalidation = xgb.DMatrix(
        X_validation,
        label=y_validation,
        feature_names=FEATURES,
    )

    start = time.perf_counter()

    model = xgb.train(
        params=parameters,
        dtrain=dtrain,
        num_boost_round=rounds,
        evals=[
            (
                dvalidation,
                "validation",
            )
        ],
        verbose_eval=20,
    )

    fit_seconds = (
        time.perf_counter() - start
    )

    prediction_start = time.perf_counter()

    probabilities = model.predict(
        dvalidation
    )

    prediction_seconds = (
        time.perf_counter()
        - prediction_start
    )

    predictions = probabilities.argmax(
        axis=1
    ).astype(np.int32)

    weighted_f1 = f1_score(
        y_validation,
        predictions,
        average="weighted",
        zero_division=0,
    )

    macro_f1 = f1_score(
        y_validation,
        predictions,
        average="macro",
        labels=np.arange(len(labels)),
        zero_division=0,
    )

    accuracy = accuracy_score(
        y_validation,
        predictions,
    )

    model_path = output_dir / "model.ubj"
    model.save_model(model_path)

    label_mapping_path = (
        output_dir / "label_mapping.json"
    )

    save_json(
        label_mapping_path,
        {
            "labels": labels,
            "label_to_index": (
                label_to_index
            ),
        },
    )

    predictions_path = (
        output_dir
        / "validation_predictions.npz"
    )

    np.savez_compressed(
        predictions_path,
        y_true=y_validation,
        y_pred=predictions,
        probabilities=probabilities,
    )

    artifact_sha256 = {
        "model.ubj": sha256_file(
            model_path
        ),
        "label_mapping.json": sha256_file(
            label_mapping_path
        ),
        "validation_predictions.npz": (
            sha256_file(predictions_path)
        ),
    }

    summary = {
        "completed": True,
        "condition": (
            "random_split__"
            "timestamp_free_plus_bandwidth"
        ),
        "model": "XGBoost",
        "seed": 42,
        "train_rows": len(X_train),
        "validation_rows": len(
            X_validation
        ),
        "feature_count": len(FEATURES),
        "features": FEATURES,
        "fit_seconds": fit_seconds,
        "validation_predict_proba_"
        "batch_seconds": prediction_seconds,
        "validation_weighted_f1": (
            weighted_f1
        ),
        "validation_macro_f1_all24": (
            macro_f1
        ),
        "validation_accuracy": accuracy,
        "serialized_model_mib": (
            model_path.stat().st_size
            / (1024 ** 2)
        ),
        "protocol_sha256": (
            protocol_sha256
        ),
        "artifact_sha256": (
            artifact_sha256
        ),
        "test_evaluated": False,
        "output_directory": str(
            output_dir
        ),
        "next": (
            "Share protocol.json and "
            "pilot_summary.json. Do not "
            "launch the remaining 11 fits."
        ),
    }

    summary_path = (
        output_dir / "pilot_summary.json"
    )

    save_json(
        summary_path,
        summary,
    )

    print()
    print(
        "XGBOOST BANDWIDTH PILOT COMPLETE"
    )
    print(
        json.dumps(
            summary,
            indent=2,
        )
    )


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
import py_compile

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

PILOT_SCRIPT = (
    ROOT
    / "bandwidth_augmentation_xgb_pilot.py"
)

assert PILOT_SCRIPT.is_file()

py_compile.compile(
    str(PILOT_SCRIPT),
    doraise=True,
)

print("PILOT SCRIPT SYNTAX CHECK PASSED")
print(PILOT_SCRIPT)

In [ ]:
import ast

source = PILOT_SCRIPT.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

required_strings = [
    "timestamp_free_plus_bandwidth",
    "random_split",
    "validation",
    "test_evaluated",
]

for value in required_strings:
    assert value in source

assert '"bandwidth"' in source
assert '"peak_time"' not in source
assert '"peak_time_ns"' not in source
assert '"start_time_ns"' not in source

print("Pilot model: XGBoost")
print("Pilot split: grouped random")
print("Pilot seed: 42")
print("Pilot feature count: 7")
print(
    "Pilot features: timestamp-free six "
    "+ bandwidth"
)
print("Hyperparameter tuning: False")
print("Test evaluation: False")
print()
print("PILOT BOUNDARY CHECK PASSED")

In [ ]:
import datetime
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = (
    datetime.datetime.now(
        datetime.timezone.utc
    )
    .strftime("%Y%m%dT%H%M%S_%fZ")
)

log_path = (
    LOG_DIR
    / (
        "bandwidth_augmentation_"
        f"xgb_pilot_{timestamp}.log"
    )
)

command = [
    sys.executable,
    "-u",
    str(
        ROOT
        / "bandwidth_augmentation_xgb_pilot.py"
    ),
    "--root",
    str(ROOT),
]

print(
    "Command:",
    " ".join(command),
)
print("Log:", log_path)
print()

environment = os.environ.copy()
environment.update(
    {
        "OMP_NUM_THREADS": "2",
        "MKL_NUM_THREADS": "2",
        "OPENBLAS_NUM_THREADS": "2",
        "VECLIB_MAXIMUM_THREADS": "2",
        "NUMEXPR_NUM_THREADS": "2",
    }
)

with log_path.open(
    "w",
    encoding="utf-8",
) as log_handle:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(
            line,
            end="",
            flush=True,
        )
        log_handle.write(line)
        log_handle.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        "Pilot stopped with exit code "
        f"{return_code}. Share the final "
        f"log lines from {log_path}"
    )

print()
print(
    "PILOT PROCESS EXITED SUCCESSFULLY"
)

In [ ]:
import json
from pathlib import Path

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

PILOT_DIR = (
    ROOT
    / "bandwidth_augmentation_v1"
    / "pilot"
    / "random_split__timestamp_free_plus_bandwidth"
    / "xgboost_seed_42"
)

required_files = [
    "protocol.json",
    "pilot_summary.json",
    "model.ubj",
    "label_mapping.json",
    "validation_predictions.npz",
]

for filename in required_files:
    path = PILOT_DIR / filename

    assert path.is_file(), (
        f"Missing pilot artifact: {path}"
    )

    print(
        f"READY: {filename} "
        f"({path.stat().st_size:,} bytes)"
    )

protocol = json.loads(
    (
        PILOT_DIR / "protocol.json"
    ).read_text(
        encoding="utf-8"
    )
)

summary = json.loads(
    (
        PILOT_DIR / "pilot_summary.json"
    ).read_text(
        encoding="utf-8"
    )
)

assert protocol["features"] == [
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

assert protocol["split"] == (
    "random_split"
)
assert protocol["seed"] == 42
assert protocol["test_evaluated"] is False

assert summary["completed"] is True
assert summary["feature_count"] == 7
assert summary["test_evaluated"] is False

print()
print("PILOT SUMMARY")
print(
    json.dumps(
        summary,
        indent=2,
    )
)
print()
print("BANDWIDTH PILOT COMPLETION CHECK PASSED")

In [ ]:
from pathlib import Path
import ast
import hashlib
import json

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


search_terms = [
    "targeted_ablation",
    "test_per_seed.csv",
    "validation_weighted_f1",
    "StandardScaler",
    "random_split__original_nine",
]

records = []

for path in sorted(ROOT.glob("*.py")):
    source = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    matched_terms = [
        term
        for term in search_terms
        if term in source
    ]

    if not matched_terms:
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    functions = [
        node.name
        for node in tree.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    ]

    classes = [
        node.name
        for node in tree.body
        if isinstance(node, ast.ClassDef)
    ]

    record = {
        "path": str(path),
        "filename": path.name,
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
        "matched_terms": matched_terms,
        "functions": functions,
        "classes": classes,
    }

    records.append(record)

print(
    "Candidate implementation files:",
    len(records),
)

for record in records:
    print()
    print("=" * 80)
    print(record["path"])
    print("SHA-256:", record["sha256"])
    print("Bytes:", record["bytes"])
    print(
        "Matched terms:",
        record["matched_terms"],
    )
    print(
        "Functions:",
        record["functions"],
    )
    print(
        "Classes:",
        record["classes"],
    )

report_path = (
    ROOT
    / "bandwidth_augmentation_v1"
    / "implementation_search.json"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

report_path.write_text(
    json.dumps(
        {
            "search_terms": search_terms,
            "candidate_count": len(records),
            "records": records,
            "training_performed": False,
            "test_evaluated": False,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", report_path)
print(
    "REPORT SHA-256:",
    sha256_file(report_path),
)

assert records, (
    "No candidate suite implementation "
    "was found."
)

print()
print(
    "IMPLEMENTATION SEARCH COMPLETE"
)
print(
    "NO TRAINING OR TEST EVALUATION "
    "WAS PERFORMED"
)

In [ ]:
implementation_report = json.loads(
    (
        ROOT
        / "bandwidth_augmentation_v1"
        / "implementation_search.json"
    ).read_text(
        encoding="utf-8"
    )
)

strong_candidates = []

for record in implementation_report[
    "records"
]:
    path = Path(record["path"])

    source = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    indicators = {
        "mentions_xgboost": (
            "XGBoost" in source
            or "xgboost" in source
        ),
        "mentions_mlp": (
            "MLP" in source
            or "mlp" in source
        ),
        "uses_three_seeds": (
            "42" in source
            and "43" in source
            and "44" in source
        ),
        "writes_test_results": (
            "test_per_seed.csv" in source
        ),
        "uses_standard_scaler": (
            "StandardScaler" in source
        ),
    }

    score = sum(indicators.values())

    if score >= 3:
        strong_candidates.append(
            {
                "path": str(path),
                "sha256": record["sha256"],
                "score": score,
                "indicators": indicators,
            }
        )

print(
    "Strong candidate files:",
    len(strong_candidates),
)

for candidate in strong_candidates:
    print()
    print(candidate["path"])
    print("SHA-256:", candidate["sha256"])
    print("Score:", candidate["score"])

    for name, status in candidate[
        "indicators"
    ].items():
        print(f"  {name}: {status}")

candidate_report_path = (
    ROOT
    / "bandwidth_augmentation_v1"
    / "suite_candidate_report.json"
)

candidate_report_path.write_text(
    json.dumps(
        {
            "candidates": strong_candidates,
            "training_performed": False,
            "test_evaluated": False,
        },
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    candidate_report_path,
)
print(
    "SHA-256:",
    sha256_file(candidate_report_path),
)
print()
print("SUITE CANDIDATE AUDIT COMPLETE")

In [ ]:
from pathlib import Path
import hashlib
import py_compile

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

SOURCE_PATH = (
    ROOT
    / "bandwidth_augmentation_xgb_pilot.py"
)

WEIGHTED_PATH = (
    ROOT
    / "bandwidth_augmentation_xgb_weighted_pilot.py"
)

source = SOURCE_PATH.read_text(
    encoding="utf-8"
)

# Add the authoritative weighting utility.
old_import = """from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
"""

new_import = """from sklearn.metrics import (
    accuracy_score,
    f1_score,
)
from sklearn.utils.class_weight import (
    compute_sample_weight,
)
"""

assert source.count(old_import) == 1
source = source.replace(
    old_import,
    new_import,
)

# Use a separate output folder so that the
# original feasibility pilot is preserved.
old_output = """/ "bandwidth_augmentation_v1"
        / "pilot"
        / "random_split__timestamp_free_plus_bandwidth"
"""

new_output = """/ "bandwidth_augmentation_v1"
        / "pilot_weighted"
        / "random_split__timestamp_free_plus_bandwidth"
"""

assert source.count(old_output) == 1
source = source.replace(
    old_output,
    new_output,
)

# Record the weighting rule in the protocol.
old_protocol = '''        "parameters": parameters,
        "rounds": rounds,
'''

new_protocol = '''        "parameters": parameters,
        "rounds": rounds,
        "class_weighting": (
            "balanced sample weights computed "
            "from the complete training partition"
        ),
'''

assert source.count(old_protocol) == 1
source = source.replace(
    old_protocol,
    new_protocol,
)

# Compute weights immediately before constructing
# the training DMatrix.
old_dmatrix = '''    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        feature_names=FEATURES,
    )
'''

new_dmatrix = '''    training_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )
        .astype(np.float32)
    )

    assert training_weights.shape == (
        len(y_train),
    )
    assert np.isfinite(
        training_weights
    ).all()
    assert (
        training_weights > 0
    ).all()

    dtrain = xgb.DMatrix(
        X_train,
        label=y_train,
        weight=training_weights,
        feature_names=FEATURES,
        nthread=2,
    )
'''

assert source.count(old_dmatrix) == 1
source = source.replace(
    old_dmatrix,
    new_dmatrix,
)

# Match the authoritative prediction DMatrix.
old_validation_dmatrix = '''    dvalidation = xgb.DMatrix(
        X_validation,
        label=y_validation,
        feature_names=FEATURES,
    )
'''

new_validation_dmatrix = '''    dvalidation = xgb.DMatrix(
        X_validation,
        label=y_validation,
        feature_names=FEATURES,
        nthread=2,
    )
'''

assert source.count(
    old_validation_dmatrix
) == 1

source = source.replace(
    old_validation_dmatrix,
    new_validation_dmatrix,
)

WEIGHTED_PATH.write_text(
    source,
    encoding="utf-8",
)

py_compile.compile(
    str(WEIGHTED_PATH),
    doraise=True,
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


print("SAVED:", WEIGHTED_PATH)
print(
    "SHA-256:",
    sha256_file(WEIGHTED_PATH),
)
print()
print(
    "CORRECTED WEIGHTED PILOT "
    "SCRIPT CREATED"
)

In [ ]:
weighted_source = (
    WEIGHTED_PATH.read_text(
        encoding="utf-8"
    )
)

required_weighting_fragments = [
    "compute_sample_weight",
    'class_weight="balanced"',
    "weight=training_weights",
    '"class_weighting"',
    '"pilot_weighted"',
]

for fragment in (
    required_weighting_fragments
):
    assert fragment in weighted_source, (
        f"Missing corrected fragment: "
        f"{fragment}"
    )

assert weighted_source.count(
    "weight=training_weights"
) == 1

assert (
    "pilot_weighted"
    in weighted_source
)

print("Model: XGBoost")
print("Split: grouped random")
print("Seed: 42")
print("Features: timestamp-free six + bandwidth")
print(
    "Class weighting: balanced training "
    "sample weights"
)
print("Hyperparameter retuning: False")
print("Test evaluation: False")
print(
    "Original unweighted pilot preserved: True"
)
print()
print(
    "WEIGHTED PILOT BOUNDARY CHECK PASSED"
)

In [ ]:
import datetime
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = (
    datetime.datetime.now(
        datetime.timezone.utc
    )
    .strftime("%Y%m%dT%H%M%S_%fZ")
)

log_path = (
    LOG_DIR
    / (
        "bandwidth_augmentation_"
        f"xgb_weighted_pilot_{timestamp}.log"
    )
)

command = [
    sys.executable,
    "-u",
    str(
        ROOT
        / "bandwidth_augmentation_xgb_weighted_pilot.py"
    ),
    "--root",
    str(ROOT),
]

environment = os.environ.copy()

environment.update(
    {
        "OMP_NUM_THREADS": "2",
        "MKL_NUM_THREADS": "2",
        "OPENBLAS_NUM_THREADS": "2",
        "VECLIB_MAXIMUM_THREADS": "2",
        "NUMEXPR_NUM_THREADS": "2",
    }
)

print("Command:", " ".join(command))
print("Log:", log_path)
print()

with log_path.open(
    "w",
    encoding="utf-8",
) as log_handle:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(
            line,
            end="",
            flush=True,
        )
        log_handle.write(line)
        log_handle.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        "Weighted pilot stopped with "
        f"exit code {return_code}. "
        f"Share the final log lines: "
        f"{log_path}"
    )

print()
print(
    "WEIGHTED PILOT PROCESS "
    "EXITED SUCCESSFULLY"
)

In [ ]:
import json
from pathlib import Path

WEIGHTED_PILOT_DIR = (
    ROOT
    / "bandwidth_augmentation_v1"
    / "pilot_weighted"
    / "random_split__timestamp_free_plus_bandwidth"
    / "xgboost_seed_42"
)

required_files = [
    "protocol.json",
    "pilot_summary.json",
    "model.ubj",
    "label_mapping.json",
    "validation_predictions.npz",
]

for filename in required_files:
    path = (
        WEIGHTED_PILOT_DIR / filename
    )

    assert path.is_file(), (
        f"Missing weighted pilot "
        f"artifact: {path}"
    )

    print(
        f"READY: {filename} "
        f"({path.stat().st_size:,} bytes)"
    )

protocol = json.loads(
    (
        WEIGHTED_PILOT_DIR
        / "protocol.json"
    ).read_text(
        encoding="utf-8"
    )
)

summary = json.loads(
    (
        WEIGHTED_PILOT_DIR
        / "pilot_summary.json"
    ).read_text(
        encoding="utf-8"
    )
)

assert protocol["class_weighting"] == (
    "balanced sample weights computed "
    "from the complete training partition"
)

assert protocol["features"] == [
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

assert summary["completed"] is True
assert summary["test_evaluated"] is False
assert summary["feature_count"] == 7

print()
print("WEIGHTED PILOT SUMMARY")
print(
    json.dumps(
        summary,
        indent=2,
    )
)
print()
print(
    "WEIGHTED PILOT COMPLETION "
    "CHECK PASSED"
)

In [ ]:
from pathlib import Path
import hashlib
import py_compile

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

FULL_SCRIPT = (
    ROOT
    / "bandwidth_augmentation_full.py"
)

EXPECTED_SHA256 = (
    "b481023112a06e10381fb756e72eb926"
    "94b07816404d32617b267b0e2fd65562"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


assert FULL_SCRIPT.is_file(), (
    f"Missing script: {FULL_SCRIPT}"
)

observed_hash = sha256_file(
    FULL_SCRIPT
)

print("Script:", FULL_SCRIPT)
print("Observed SHA-256:", observed_hash)
print("Expected SHA-256:", EXPECTED_SHA256)

assert observed_hash == EXPECTED_SHA256

py_compile.compile(
    str(FULL_SCRIPT),
    doraise=True,
)

print()
print(
    "FULL-SUITE SCRIPT HASH AND "
    "SYNTAX VERIFIED"
)

In [ ]:
source = FULL_SCRIPT.read_text(
    encoding="utf-8"
)

required_fragments = [
    '"expected_fits": 12',
    '"pilot_reuse": False',
    '"selection_used_outer_validation"',
    '"timestamp_free_plus_bandwidth"',
    '"random_split"',
    '"chronological_split"',
    'compute_sample_weight("balanced"',
    "StandardScaler",
    "test_per_seed.csv",
    "paired_differences.csv",
]

for fragment in required_fragments:
    assert fragment in source, (
        f"Missing protocol element: "
        f"{fragment}"
    )

assert source.count(
    "for seed in SEEDS"
) == 1

print("Models: XGBoost and MLP")
print(
    "Splits: grouped-random and "
    "chronological"
)
print("Seeds: 42, 43, 44")
print("Total final fits: 12")
print(
    "Features: timestamp-free six "
    "+ bandwidth"
)
print(
    "Configurations: inherited from "
    "split-specific timestamp-free searches"
)
print("Bandwidth-specific retuning: False")
print("Pilot reuse: False")
print(
    "XGBoost weighting: balanced "
    "training-sample weights"
)
print(
    "MLP preprocessing: training-fitted "
    "StandardScaler"
)
print(
    "Final test policy: one evaluation "
    "per frozen model/split/seed"
)
print()
print(
    "FULL-SUITE EXPERIMENTAL "
    "BOUNDARY VERIFIED"
)

In [ ]:
import datetime
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

LOG_DIR = ROOT / "run_logs"
LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = (
    datetime.datetime.now(
        datetime.timezone.utc
    )
    .strftime("%Y%m%dT%H%M%S_%fZ")
)

log_path = (
    LOG_DIR
    / (
        "bandwidth_augmentation_"
        f"full_{timestamp}.log"
    )
)

command = [
    sys.executable,
    "-u",
    str(
        ROOT
        / "bandwidth_augmentation_full.py"
    ),
    "--root",
    str(ROOT),
]

environment = os.environ.copy()

environment.update(
    {
        "OMP_NUM_THREADS": "2",
        "MKL_NUM_THREADS": "2",
        "OPENBLAS_NUM_THREADS": "2",
        "VECLIB_MAXIMUM_THREADS": "2",
        "NUMEXPR_NUM_THREADS": "2",
    }
)

print("Command:", " ".join(command))
print("Log:", log_path)
print()
print(
    "The run is sequential and "
    "restart-safe."
)
print(
    "Do not run other memory-intensive "
    "programs simultaneously."
)
print()

with log_path.open(
    "w",
    encoding="utf-8",
) as log_handle:
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(
            line,
            end="",
            flush=True,
        )
        log_handle.write(line)
        log_handle.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        "Full suite stopped with exit "
        f"code {return_code}. Share the "
        f"final log lines from {log_path}"
    )

print()
print(
    "FULL-SUITE PROCESS "
    "EXITED SUCCESSFULLY"
)

In [ ]:
import json
import pandas as pd
from pathlib import Path

SUITE_DIR = (
    ROOT
    / "bandwidth_augmentation_v1"
    / "full_suite"
)

required_files = [
    "protocol.json",
    "summary.json",
    "test_per_seed.csv",
    "test_summary.csv",
    "paired_differences.csv",
    "paired_difference_summary.csv",
]

for filename in required_files:
    path = SUITE_DIR / filename

    assert path.is_file(), (
        f"Missing output: {path}"
    )

    print(
        f"READY: {filename} "
        f"({path.stat().st_size:,} bytes)"
    )

summary = json.loads(
    (
        SUITE_DIR / "summary.json"
    ).read_text(
        encoding="utf-8"
    )
)

per_seed = pd.read_csv(
    SUITE_DIR / "test_per_seed.csv"
)

test_summary = pd.read_csv(
    SUITE_DIR / "test_summary.csv"
)

differences = pd.read_csv(
    SUITE_DIR / "paired_differences.csv"
)

difference_summary = pd.read_csv(
    SUITE_DIR
    / "paired_difference_summary.csv"
)

assert summary["completed"] is True
assert summary["workers"] == 12
assert summary["pilot_reused"] is False
assert summary["test_evaluated"] is True

assert len(per_seed) == 12
assert len(test_summary) == 4
assert len(differences) == 12
assert len(difference_summary) == 4

assert set(per_seed["model"]) == {
    "XGBoost",
    "MLP",
}

assert set(per_seed["split"]) == {
    "random_split",
    "chronological_split",
}

assert set(per_seed["seed"]) == {
    42,
    43,
    44,
}

print()
print("FINAL TEST SUMMARY")
display(test_summary)

print()
print(
    "PAIRED SIX-VERSUS-SEVEN-"
    "FEATURE DIFFERENCES"
)
display(difference_summary)

print()
print("FINAL SUITE RECORD")
print(
    json.dumps(
        summary,
        indent=2,
    )
)

print()
print(
    "BANDWIDTH-AUGMENTATION "
    "SUITE COMPLETION CHECK PASSED"
)